In [24]:
import io
import chess.pgn
import zstandard
from itertools import islice
from pathlib import Path
import time
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path = ROOT / "data" / "lichess_db_standard_rated_2015-06.pgn"
def iter_games(path):
    path = Path(path)
    if path.suffix == ".zst":
        fh = open(path, "rb")
        reader = zstandard.ZstdDecompressor().stream_reader(fh)
        stream = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")
    else:
        stream = open(path, encoding="utf-8", errors="replace")
    with stream:
        while True:
            game = chess.pgn.read_game(stream)
            if game is None:
                return
            yield game

In [18]:
seuilElo = 1900

def get_elo(headers, key):
    v = headers.get(key, "")
    return int(v) if v.isdigit() else None
def keep_game(game):
    white_elo = get_elo(game.headers, "WhiteElo")
    black_elo = get_elo(game.headers, "BlackElo")
    return (min(white_elo, black_elo) >= seuilElo)


In [19]:
for g in islice(iter_games(path), 3):
    print(dict(g.headers))
    print(keep_game(g))
    print("---")

{'Event': 'Rated Blitz tournament https://lichess.org/tournament/UtqbCHBF', 'Site': 'https://lichess.org/1r6oKss8', 'Date': '????.??.??', 'Round': '?', 'White': 'reydeguerras', 'Black': 'bishop50', 'Result': '0-1', 'UTCDate': '2015.05.31', 'UTCTime': '22:00:01', 'WhiteElo': '1534', 'BlackElo': '1462', 'ECO': '?', 'Opening': '?', 'TimeControl': '300+0', 'Termination': 'Abandoned'}
False
---
{'Event': 'Rated Blitz tournament https://lichess.org/tournament/UtqbCHBF', 'Site': 'https://lichess.org/Ct9uhSBp', 'Date': '????.??.??', 'Round': '?', 'White': 'penpalrdro', 'Black': 'brumia', 'Result': '1-0', 'UTCDate': '2015.05.31', 'UTCTime': '22:00:01', 'WhiteElo': '1598', 'BlackElo': '1621', 'ECO': 'A10', 'Opening': 'English Opening', 'TimeControl': '300+0', 'Termination': 'Abandoned'}
False
---
{'Event': 'Rated Blitz game', 'Site': 'https://lichess.org/PezGX4sx', 'Date': '????.??.??', 'Round': '?', 'White': 'Fumodilondra', 'Black': 'IsraelAlcaraz', 'Result': '0-1', 'UTCDate': '2015.05.31', 'UT

In [20]:
import time
from itertools import islice

kept = total = 0
t0 = time.time()

for g in islice(iter_games(path), 50_000):
    total += 1
    if keep_game(g):
        kept += 1

dt = time.time() - t0
print(f"{kept}/{total} = {kept/total:.1%}")
print(f"{total/dt:.0f} parties/s")

3740/50000 = 7.5%
531 parties/s


In [32]:
import numpy as np
from chessai.encoding import board_to_tensor, move_to_index, index_to_move

RESULT_TO_VALUE = {"1-0": 1, "0-1": -1, "1/2-1/2": 0}

SKIP_OPENING = 10       # demi-coups sautés en début de partie
SAMPLES_PER_GAME = 25   # positions gardées par partie


def extract_positions(game, rng):
    value = RESULT_TO_VALUE.get(game.headers.get("Result"))
    if value is None:
        return

    moves = list(game.mainline_moves())
    n = len(moves)
    if n <= SKIP_OPENING:
        return

    candidates = np.arange(SKIP_OPENING, n)
    if len(candidates) > SAMPLES_PER_GAME:
        candidates = rng.choice(candidates, SAMPLES_PER_GAME, replace=False)
    wanted = set(int(i) for i in candidates)

    board = game.board()
    for i, move in enumerate(moves):
        if i in wanted:
            tensor = board_to_tensor(board).numpy().astype(np.uint8)
            yield tensor, move_to_index(move), value
        board.push(move)

In [ ]:
from itertools import islice

n = 0
rng = np.random.default_rng(0)

for g in islice(iter_games(path), 200):
    if not keep_game(g):
        continue
    board = g.board()
    for t, idx, v in extract_positions(g, rng):
        assert 0 <= idx < 4672
        assert v in (-1, 0, 1)
        n += 1
print(n, "positions")

407 positions


In [25]:
def build_dataset(path, out_dir, max_games=None, seed=0):
    rng = np.random.default_rng(seed)
    xs, ps, vs = [], [], []

    games = iter_games(path)
    if max_games is not None:
        games = islice(games, max_games)

    for total, game in enumerate(games):
        if total % 10000 == 0:
            print(total, len(xs))
        if not keep_game(game):
            continue
        for tensor, idx, value in extract_positions(game, rng):
            xs.append(tensor)
            ps.append(idx)
            vs.append(value)
        

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "X.npy", np.stack(xs))
    np.save(out_dir / "policy.npy", np.array(ps, dtype=np.int16))
    np.save(out_dir / "value.npy", np.array(vs, dtype=np.int8))
    print(len(xs), "positions")

In [26]:
build_dataset(path, "data/small", max_games=5000)

0 0
8819 positions


In [27]:
import numpy as np

X = np.load("data/small/X.npy")
p = np.load("data/small/policy.npy")
v = np.load("data/small/value.npy")

print(X.shape, X.dtype)
print(p.min(), p.max())
print(np.bincount(v + 1))                      # [noirs, nulles, blancs]
print(X[:, :12].sum(axis=(1, 2, 3))[:20])      # pièces par position

(8819, 18, 8, 8) uint8
1 4653
[3408  575 4836]
[30 30 30 30 29 28 28 27 26 26 26 25 25 25 24 23 22 20 20 20]


In [30]:
s = 0
tot = 0
for g in islice(iter_games(path), 5000):
    if keep_game(g):
        tot+=1
        if g.headers["Result"] == "1/2-1/2":
            s+=1
print(s/tot)

0.06575342465753424


In [33]:
rng = np.random.default_rng(0)
g = next(x for x in islice(iter_games(path), 200) if keep_game(x))
board = g.board()
moves = list(g.mainline_moves())
for i, m in enumerate(moves):
    if i == 20:
        idx = move_to_index(m)
        assert index_to_move(idx, board) in board.legal_moves
        print("ok", board.san(m))
        break
    board.push(m)

ok Bg5


In [ ]:
kept = total = 0
for g in islice(iter_games(path), 50_000):
    total += 1
    kept += keep_game(g)
print(f"{kept}/{total} = {kept/total:.1%}")